# Week 10 — LLM-mediated interpersonal messages

**Research task:** Compare how one model formulates the same broken commitment under two instructions and for three relationships. We observe generated writing and researcher notes. We do not observe sincerity, persuasion or relationship repair.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjbarrie/GenAI_Soc2026/blob/main/workbook/session10/session10_relationship_mediation.ipynb)

## Prepare the notebook environment
Run the next cell first. Colab installs the two SDKs and uses OpenRouter. Local JupyterLab checks the course environment and uses Ollama by default.

In [ ]:
# colab-setup
SESSION = "session10"

# Run this cell first. It prepares Colab or checks the local Python environment.
import importlib as setup_importlib
import importlib.util as setup_importlib_util
import os as setup_os
import subprocess as setup_subprocess
import sys as setup_sys
from pathlib import Path as SetupPath

try:
    import google.colab as setup_colab  # type: ignore[import-not-found]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

course_packages = {
    "openrouter": "openrouter>=0.6,<1",
    "ollama": "ollama>=0.6,<1",
}
if SESSION == "session05":
    course_packages["plotly"] = "plotly>=6.3,<7"
    course_packages["umap"] = "umap-learn>=0.5.9,<1"
    course_packages["pandas"] = "pandas>=2.3,<3"
if SESSION == "session13":
    course_packages["pandas"] = "pandas>=2.2,<3"

missing_packages = [
    package_name
    for package_name in course_packages
    if setup_importlib_util.find_spec(package_name) is None
]

if IN_COLAB:
    if missing_packages:
        packages_to_install = [course_packages[name] for name in missing_packages]
        setup_subprocess.run(
            [
                setup_sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--disable-pip-version-check",
                *packages_to_install,
            ],
            check=True,
        )
        setup_importlib.invalidate_caches()

    COURSE_ROOT = SetupPath(
        setup_os.getenv("COURSE_COLAB_ROOT", "/content/GenAI_Soc2026")
    )
    if not COURSE_ROOT.exists():
        setup_subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/cjbarrie/GenAI_Soc2026.git",
                str(COURSE_ROOT),
            ],
            check=True,
        )
    setup_os.chdir(COURSE_ROOT / "workbook" / SESSION)
else:
    COURSE_ROOT = SetupPath.cwd()
    while not (COURSE_ROOT / "config" / "course_models.json").exists() and COURSE_ROOT != COURSE_ROOT.parent:
        COURSE_ROOT = COURSE_ROOT.parent
    if missing_packages:
        missing_text = ", ".join(missing_packages)
        raise ModuleNotFoundError(
            f"This notebook is using a Python environment without: {missing_text}.\n\n"
            "Close Jupyter. Open a terminal in the GenAI_Soc2026 repository and run:\n"
            "    uv sync\n"
            "    uv run jupyter lab\n\n"
            "In VS Code, select the Python interpreter inside the repository's .venv folder."
        )
    if not (COURSE_ROOT / "config" / "course_models.json").exists():
        raise FileNotFoundError(
            "The complete GenAI_Soc2026 repository could not be found. A notebook "
            "downloaded by itself is not enough for local work. Download or clone the "
            "repository, open a terminal in that folder, and run: uv run jupyter lab"
        )

course_root_text = str(COURSE_ROOT)
if course_root_text not in setup_sys.path:
    setup_sys.path.insert(0, course_root_text)

still_missing = [
    package_name
    for package_name in course_packages
    if setup_importlib_util.find_spec(package_name) is None
]
if still_missing:
    raise ModuleNotFoundError(
        "Setup did not make these packages available: " + ", ".join(still_missing)
    )

print("Environment:", "Google Colab" if IN_COLAB else "local course environment")
print("Python executable:", setup_sys.executable)
print("Course root:", COURSE_ROOT)
print("Working folder:", SetupPath.cwd())
print("Python SDKs: ready")
if IN_COLAB:
    print("Route for this runtime: OpenRouter")
    print("Ollama work: complete later in local JupyterLab or on the in-class machine")
else:
    import json as setup_json
    import ollama as setup_ollama

    setup_config = setup_json.loads(
        (COURSE_ROOT / "config" / "course_models.json").read_text()
    )
    setup_local_model = setup_config["local"]["model"]
    try:
        setup_models = setup_ollama.list().models
        setup_model_names = [
            getattr(item, "model", None) or getattr(item, "name", None)
            for item in setup_models
        ]
        print("Ollama server: reachable at localhost:11434")
        if setup_local_model in setup_model_names:
            print("Course local model: ready —", setup_local_model)
        else:
            print("Course local model: NOT INSTALLED —", setup_local_model)
            print("NEXT STEP: open a terminal and run: ollama pull " + setup_local_model)
    except Exception as setup_error:
        print("Ollama server: NOT REACHABLE")
        print("NEXT STEP: start the Ollama application, then run: ollama list")
        print("Diagnostic:", str(setup_error).splitlines()[0])


## Import the SDKs and select a route
The imports load installed tools. `ROUTE` is a string. Colab selects the hosted route; local JupyterLab selects the local route.

In [ ]:
import json
import os
from getpass import getpass
import ollama
from openrouter import OpenRouter

config = json.loads((COURSE_ROOT / "config" / "course_models.json").read_text())
OPENROUTER_MODEL = config["hosted"]["model"]
OLLAMA_MODEL = config["local"]["model"]
HOSTED_MODEL = OPENROUTER_MODEL
LOCAL_MODEL = OLLAMA_MODEL
ROUTE = "openrouter" if IN_COLAB else "ollama"  # local default
USE_RECORDED = False
print("Selected route:", ROUTE)


## Store the facts and the two instructions
`facts` is a dictionary. Each instruction is a string. The facts contain no supplied feelings, motives or relationship history.

In [ ]:
facts = {
    "commitment": "help with a move at 8 a.m. Saturday",
    "work_conflict_known": "noon Friday",
    "message_sent": "9 p.m. Friday",
    "reliance": "the recipient declined paid help because of the promise",
    "repairs": ["pay half the cost of a last-minute mover", "help on Sunday"],
}
thin_instruction = ("Write a message telling this person that I cannot help them move tomorrow. "
                    "Use only the supplied facts and do not invent facts.")
obligation_instruction = ("Write the message to this person. State the commitment that was broken, "
    "acknowledge the late notice and the plan they made in reliance on the promise, "
    "explain only the supplied facts, and offer both available forms of repair. "
    "Do not invent feelings, relationship history or motives.")
print(type(facts), type(thin_instruction), type(obligation_instruction))


## Define one function
The function receives a relationship string, an instruction-name string and a route string. It constructs the complete prompt, shows the route-specific call and returns one raw string.

In [ ]:
recorded_returns = {
    ("thin", "close friend"): "I’m sorry, but I can’t help tomorrow. I can help Sunday or pay half a mover.",
    ("obligation", "colleague"): "I broke my commitment and gave you late notice. I can pay half a mover or help Sunday.",
    ("obligation", "close friend"): "I’m sorry. I broke my promise and told you late after you declined paid help. I can pay half a mover or help Sunday.",
    ("obligation", "romantic partner"): "I broke my promise and waited too long to tell you after you relied on me. I can pay half a mover or help Sunday.",
}

def draft_message(relationship, instruction_name, route):
    if instruction_name == "thin":
        instruction = thin_instruction
    elif instruction_name == "obligation":
        instruction = obligation_instruction
    else:
        raise ValueError("instruction_name must be 'thin' or 'obligation'")
    prompt = f"""{instruction}\n\nRelationship: {relationship}\n\nFacts:\n{json.dumps(facts, indent=2)}\n\nReturn only the message that could be sent."""
    print("Inputs:", relationship, instruction_name, route)
    print("Prompt:\n", prompt)
    if USE_RECORDED:
        raw_output = recorded_returns[(instruction_name, relationship)]
    elif route == "openrouter":
        api_key = os.getenv("OPENROUTER_API_KEY") or getpass("OpenRouter course key (hidden): " )
        with OpenRouter(api_key=api_key) as client:
            response = client.chat.send(model=OPENROUTER_MODEL, messages=[{"role": "user", "content": prompt}], temperature=0)
        raw_output = response.choices[0].message.content
    elif route == "ollama":
        response = ollama.chat(think=False, model=OLLAMA_MODEL, messages=[{"role": "user", "content": prompt}], options={"temperature": 0})
        raw_output = response.message.content
    else:
        raise ValueError("route must be 'openrouter' or 'ollama'")
    print("Raw return:\n", raw_output)
    print("Output type:", type(raw_output))
    return raw_output


## Compare the two instructions
The relationship remains `close friend`. Only the instruction changes. Read both messages and note what the thin prompt leaves the model to infer.

In [ ]:
thin_message = draft_message("close friend", "thin", ROUTE)
obligation_message = draft_message("close friend", "obligation", ROUTE)
print("\nTHIN INSTRUCTION\n", thin_message)
print("\nRELATIONAL-OBLIGATION INSTRUCTION\n", obligation_message)


## Repeat the same routine across three relationships
On the first iteration, `relationship` contains `"colleague"`. The loop appends one dictionary to `records` after each raw return.

In [ ]:
relationships = ["colleague", "close friend", "romantic partner"]
records = []
for relationship in relationships:
    message = draft_message(relationship, "obligation", ROUTE)
    records.append({"relationship": relationship, "instruction": "obligation", "facts": facts.copy(), "raw_message": message})

for record in records:
    print("\nRELATIONSHIP:", record["relationship"])
    print(record["raw_message"])


## Keep researcher notes separate
Fill these lists after reading the outputs. Do not ask the model to decide whether its own message is sincere, appropriate or effective.

In [ ]:
researcher_notes = {
    "invented_facts": [],
    "commitment_named": [],
    "late_notice_named": [],
    "reliance_named": [],
    "repairs_offered": [],
    "other_observations": [],
}
print(json.dumps(researcher_notes, indent=2))


## Your one change and recording
Change one relationship description or one obligation, rerun, and explain what changed in the generated writing. State explicitly what the exercise cannot establish about the sender, recipient or relationship. Follow the Week 10 page for the 6–8 minute recording checklist.